# 06: Union mediation

Author's model (`refs/author_spec.md`, "Union mediation" and "Union premium by outsourcing status"). All union-mediation
code (estimation and tables) lives here, not in `02_estimates` or `03_tables`.

Samples: 5-year bins 1983-1987, 1988-1992, 1993-1997, 1998-2000, 2001-2005, 2006-2010, 2011-2015, 2016-2019, 2021-2025.
1998-2000 and 2016-2019 are shorter bins (3 and 4 years); 2020 is excluded from every bin.

Two decompositions, both descriptive, not causal:
1. **Mediation**: `mediated_penalty = (u_inhouse - u_outsourced) * beta_union`, with an approximate delta-method SE,
   and `pct_explained = mediated_penalty / |beta_outsourced|` from the `nounion` model. Base, nounion, and the union
   rates use identical rows in every bin.
2. **Union premium by outsourcing status**: the interactions model (D&K row 4) in each bin, reporting `outsourced`,
   `union`, and `union_x_outsourced`. A negative `union_x_outsourced` coefficient means a smaller estimated union
   premium for outsourced workers in that bin.

Outputs: `outputs/py/mediation.csv`, `outputs/py/union_interactions.csv` (all 9 bins; cells 1-5 refit these and
are only rerun when the cleaning or model changes). The table cells below slice those same CSVs, with no
re-estimation, into:
- **Table 4** (body): 2001-2025 bins only, union-gap component shown as negative, plus 2001-2005-to-2021-2025 change
  rows (`table4_union_mediation.{csv,tex}`).
- **Table A6** (appendix): 1983-2000 bins, original sign convention (`tableA6_union_mediation_1983_2000.{csv,tex}`).
- **Table A5** (appendix): union x outsourced interactions, panels for 2001-2025 and 1983-2000 bins
  (`tableA5_union_premium_interactions.{csv,tex}`).

No figure: the previous union-mediation figure and its file are removed.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import pyfixest as pf

OUT = "../outputs"

author_terms = ["log_wage", "outsourced", "union", "parttime", "educ_cat", "female", "black", "hispanic",
                "age", "age2", "msa", "central_city", "cc_unknown", "geo_unidentified"]
nounion_terms = [t for t in author_terms if t != "union"]
author_formula = (
    "log_wage ~ outsourced + union + parttime + educ_cat + female + black + hispanic "
    "+ age + age2 + msa + central_city + cc_unknown + geo_unidentified | STATEFIP^YEAR"
)
nounion_formula = author_formula.replace("outsourced + union + parttime", "outsourced + parttime")
interact_formula = author_formula.replace(
    " | STATEFIP^YEAR", " + union_x_outsourced + parttime_x_outsourced | STATEFIP^YEAR")

cols = (["occupation", "YEAR", "outsourced", "EARNWT", "ym", "STATEFIP",
        "union_x_outsourced", "parttime_x_outsourced"] + author_terms)
dk_clean = pd.read_parquet(f"{OUT}/py/dk_clean.parquet", columns=list(dict.fromkeys(cols)))

# 5-year bins (refs/author_spec.md, "Union mediation"); True marks the shorter bins
BINS5 = [("1983-1987", 1983, 1987, False), ("1988-1992", 1988, 1992, False), ("1993-1997", 1993, 1997, False),
         ("1998-2000", 1998, 2000, True), ("2001-2005", 2001, 2005, False), ("2006-2010", 2006, 2010, False),
         ("2011-2015", 2011, 2015, False), ("2016-2019", 2016, 2019, True), ("2021-2025", 2021, 2025, False)]
OCC_LABEL = {"janitor": "Janitors", "guard": "Guards"}

def weighted_rate_se(x, w):
    # EARNWT-weighted share and its SE, Kish effective N (same formula as incidence in 02_estimates)
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    p = np.sum(w * x) / np.sum(w)
    n_eff = (np.sum(w) ** 2) / np.sum(w ** 2)
    return p, np.sqrt(p * (1 - p) / n_eff)

print("bins:", len(BINS5), "| rows:", len(dk_clean))

## 1. Mediation: gap, union premium, mediated penalty (delta-method SE)

In [ ]:
med_rows = []
for occ in ["janitor", "guard"]:
    for label, y0, y1, short in BINS5:
        s = dk_clean[(dk_clean["occupation"] == occ) & dk_clean["YEAR"].between(y0, y1)].dropna(subset=author_terms).copy()

        base = pf.feols(author_formula, data=s, weights="EARNWT", vcov={"CRV1": "ym"})
        beta_union, se_beta = base.tidy().loc["union", ["Estimate", "Std. Error"]]

        out_mask = s["outsourced"] == 1
        in_mask = s["outsourced"] == 0
        u_out, se_out = weighted_rate_se(s.loc[out_mask, "union"], s.loc[out_mask, "EARNWT"])
        u_in, se_in = weighted_rate_se(s.loc[in_mask, "union"], s.loc[in_mask, "EARNWT"])
        gap = u_in - u_out
        se_gap = math.sqrt(se_in**2 + se_out**2)  # independent samples

        nounion = pf.feols(nounion_formula, data=s.dropna(subset=nounion_terms), weights="EARNWT", vcov={"CRV1": "ym"})
        assert nounion._N == base._N, (occ, label, "nounion N differs from base N", nounion._N, base._N)
        raw_penalty, se_raw = nounion.tidy().loc["outsourced", ["Estimate", "Std. Error"]]

        mediated = gap * beta_union
        se_mediated = math.sqrt((beta_union**2) * (se_gap**2) + (gap**2) * (se_beta**2))  # delta method
        pct_explained = mediated / abs(raw_penalty)

        med_rows.append(dict(occ=occ, period=label, short_bin=short,
                             u_inhouse=u_in, se_u_inhouse=se_in, u_outsourced=u_out, se_u_outsourced=se_out,
                             gap=gap, se_gap=se_gap, beta_union=beta_union, se_beta_union=se_beta,
                             raw_penalty_nounion=raw_penalty, se_raw_penalty_nounion=se_raw,
                             mediated_penalty=mediated, se_mediated_penalty=se_mediated,
                             pct_explained=pct_explained, n_obs=base._N))

mediation = pd.DataFrame(med_rows)
mediation.to_csv(f"{OUT}/py/mediation.csv", index=False)
print(mediation.round(4).to_string(index=False))

## 2. Union premium by outsourcing status (interactions model, D&K row 4)

Same sample `s` as the mediation base model in each bin (same dropna on `author_terms`; `union_x_outsourced` and
`parttime_x_outsourced` are products of columns already required non-missing, so no additional rows are dropped).

In [ ]:
int_rows = []
for occ in ["janitor", "guard"]:
    for label, y0, y1, short in BINS5:
        s = dk_clean[(dk_clean["occupation"] == occ) & dk_clean["YEAR"].between(y0, y1)].dropna(subset=author_terms).copy()
        fit = pf.feols(interact_formula, data=s, weights="EARNWT", vcov={"CRV1": "ym"})
        tidy = fit.tidy()
        for term in ["outsourced", "union", "union_x_outsourced"]:
            r = tidy.loc[term]
            int_rows.append(dict(occ=occ, period=label, short_bin=short, term=term,
                                 estimate=r["Estimate"], se=r["Std. Error"], p=r["Pr(>|t|)"], n_obs=fit._N))

interactions = pd.DataFrame(int_rows)
interactions.to_csv(f"{OUT}/py/union_interactions.csv", index=False)
print(interactions.round(4).to_string(index=False))

## Remove the old union-mediation figure

Replaced by Table 4 below.

In [ ]:
fig_path = f"{OUT}/figures/union_mediation.png"
if os.path.exists(fig_path):
    os.remove(fig_path)
    print("removed", fig_path)
else:
    print("no figure file present")

## Table 4 (body): union mediation, 2001-2025, by 5-year bin

Union-gap component shown as negative (= -mediated_penalty). Change rows compare 2001-2005 to 2021-2025, with SEs from independent bins.

In [ ]:
BIN_ORDER = [b[0] for b in BINS5]
BODY_LABELS = [b[0] for b in BINS5 if int(b[0][:4]) >= 2001]  # 2001-2005 .. 2021-2025

med_body = mediation[mediation["period"].isin(BODY_LABELS)].copy()
med_body["union_gap_component"] = -med_body["mediated_penalty"]
med_body["se_union_gap_component"] = med_body["se_mediated_penalty"]

NOTE4 = (r"Gap = union rate, in-house minus outsourced, in percentage points. Union premium ($\beta$) is the union "
         r"coefficient from the author's base model. Raw penalty is the outsourced coefficient from the nounion model "
         r"(no union control). Union-gap component (log points) = $-$(gap $\times$ union premium), with an "
         r"approximate delta-method SE: sqrt($\beta^2$ var(gap) + gap$^2$ var($\beta$)); shown as negative, in the "
         r"same direction as the raw penalty. \% explained = $|$union-gap component$|$ / $|$raw penalty$|$, and is "
         r"unstable where the raw penalty is small. The change row compares 2001-2005 to 2021-2025 (independent "
         r"bins, SE = sqrt(SE$_1^2$ + SE$_2^2$)). SEs in parentheses. Descriptive decomposition, not causal. "
         r"Data: CPS-ORG via IPUMS.")

def plabel(period, short):
    return period + (r"$^\dagger$" if short else "")

def row4(r):
    cells = [plabel(r["period"], r["short_bin"]),
             f"{100 * r['u_inhouse']:.1f}\\%",
             f"{100 * r['u_outsourced']:.1f}\\%",
             f"{100 * r['gap']:.1f}pp",
             f"{r['beta_union']:.3f} ({r['se_beta_union']:.3f})",
             f"{r['raw_penalty_nounion']:.3f} ({r['se_raw_penalty_nounion']:.3f})",
             f"{r['union_gap_component']:.3f} ({r['se_union_gap_component']:.3f})",
             f"{100 * r['pct_explained']:.1f}\\%",
             f"{int(r['n_obs']):,}"]
    return " & ".join(cells) + r" \\"

def change_row(occ):
    r1 = med_body[(med_body["occ"] == occ) & (med_body["period"] == "2001-2005")].iloc[0]
    r2 = med_body[(med_body["occ"] == occ) & (med_body["period"] == "2021-2025")].iloc[0]
    import math
    d_raw = r2["raw_penalty_nounion"] - r1["raw_penalty_nounion"]
    se_d_raw = math.sqrt(r1["se_raw_penalty_nounion"]**2 + r2["se_raw_penalty_nounion"]**2)
    d_comp = r2["union_gap_component"] - r1["union_gap_component"]
    se_d_comp = math.sqrt(r1["se_union_gap_component"]**2 + r2["se_union_gap_component"]**2)
    cells = [r"\textit{Change, 2001-2005 to 2021-2025}", "", "", "", "",
             f"{d_raw:+.3f} ({se_d_raw:.3f})", f"{d_comp:+.3f} ({se_d_comp:.3f})", "", ""]
    return " & ".join(cells) + r" \\"

header4 = ["Period", "Union Rate In-House", "Union Rate Outsourced", "Gap", r"Union Premium ($\beta$)",
           "Raw Penalty", "Union-Gap Component (log points)", r"\% Explained", "N"]
col = r">{\centering\arraybackslash}p{1.55cm}"
body4 = [r"\toprule", " & ".join(header4) + r" \\", r"\midrule"]
for occ, occ_label in [("janitor", "Janitors"), ("guard", "Guards")]:
    body4.append(r"\multicolumn{9}{l}{\textbf{" + occ_label + r"}} \\")
    sub = med_body[med_body["occ"] == occ].set_index("period").loc[BODY_LABELS].reset_index()
    for _, r in sub.iterrows():
        body4.append(row4(r))
    body4.append(change_row(occ))
    if occ == "janitor":
        body4.append(r"\addlinespace")
body4.append(r"\bottomrule")

tex4 = "\n".join([
    r"\begin{table}[!t]", r"\centering",
    r"\caption{Table 4: Union Mediation of the Outsourcing Wage Penalty, 2001-2025}",
    r"\begin{tabular}{l" + col * 8 + "}",
    *body4,
    r"\end{tabular}",
    r"\begin{minipage}{\linewidth}\footnotesize", NOTE4, r"\end{minipage}",
    r"\end{table}", ""])
with open(f"{OUT}/tables/table4_union_mediation.tex", "w") as f:
    f.write(tex4)
med_body.to_csv(f"{OUT}/tables/table4_union_mediation.csv", index=False)
print(tex4)

## Table A6 (appendix): union mediation, 1983-2000, by 5-year bin

Original sign convention (mediated penalty shown as positive).

In [ ]:
APPX_LABELS = [b[0] for b in BINS5 if int(b[0][:4]) < 2001]  # 1983-1987 .. 1998-2000
med_appx = mediation[mediation["period"].isin(APPX_LABELS)].copy()
med_appx["union_gap_component"] = -med_appx["mediated_penalty"]
med_appx["se_union_gap_component"] = med_appx["se_mediated_penalty"]

NOTE_A6 = (r"Gap = union rate, in-house minus outsourced, in percentage points. Union premium ($\beta$) is the union "
           r"coefficient from the author's base model. Raw penalty is the outsourced coefficient from the nounion "
           r"model (no union control). Union-gap component (log points) = $-$(gap $\times$ union premium), with an "
           r"approximate delta-method SE: sqrt($\beta^2$ var(gap) + gap$^2$ var($\beta$)); shown as negative, in the "
           r"same direction as the raw penalty (same convention as Table 4). \% explained = $|$union-gap "
           r"component$|$ / $|$raw penalty$|$, and is unstable where the raw penalty is small. SEs in parentheses. "
           r"$^\dagger$1998-2000 is a shorter bin (3 years). Descriptive decomposition, not causal. "
           r"Data: CPS-ORG via IPUMS.")

def rowA6(r):
    cells = [plabel(r["period"], r["short_bin"]),
             f"{100 * r['u_inhouse']:.1f}\\%",
             f"{100 * r['u_outsourced']:.1f}\\%",
             f"{100 * r['gap']:.1f}pp",
             f"{r['beta_union']:.3f} ({r['se_beta_union']:.3f})",
             f"{r['raw_penalty_nounion']:.3f} ({r['se_raw_penalty_nounion']:.3f})",
             f"{r['union_gap_component']:.3f} ({r['se_union_gap_component']:.3f})",
             f"{100 * r['pct_explained']:.1f}\\%",
             f"{int(r['n_obs']):,}"]
    return " & ".join(cells) + r" \\"

headerA6 = ["Period", "Union Rate In-House", "Union Rate Outsourced", "Gap", r"Union Premium ($\beta$)",
            "Raw Penalty", "Union-Gap Component (log points)", r"\% Explained", "N"]
bodyA6 = [r"\toprule", " & ".join(headerA6) + r" \\", r"\midrule"]
for occ, occ_label in [("janitor", "Janitors"), ("guard", "Guards")]:
    bodyA6.append(r"\multicolumn{9}{l}{\textbf{" + occ_label + r"}} \\")
    sub = med_appx[med_appx["occ"] == occ].set_index("period").loc[APPX_LABELS].reset_index()
    for _, r in sub.iterrows():
        bodyA6.append(rowA6(r))
    if occ == "janitor":
        bodyA6.append(r"\addlinespace")
bodyA6.append(r"\bottomrule")

texA6 = "\n".join([
    r"\begin{table}[!t]", r"\centering",
    r"\caption{Table A6: Union Mediation of the Outsourcing Wage Penalty, 1983-2000}",
    r"\begin{tabular}{l" + col * 8 + "}",
    *bodyA6,
    r"\end{tabular}",
    r"\begin{minipage}{\linewidth}\footnotesize", NOTE_A6, r"\end{minipage}",
    r"\end{table}", ""])
with open(f"{OUT}/tables/tableA6_union_mediation_1983_2000.tex", "w") as f:
    f.write(texA6)
med_appx.to_csv(f"{OUT}/tables/tableA6_union_mediation_1983_2000.csv", index=False)
print(texA6)

## Table A5 (appendix): union premium by outsourcing status, by panel

In [ ]:
PANELS_A5 = [("2001-2025", [b for b in BINS5 if int(b[0][:4]) >= 2001]),
             ("1983-2000", [b for b in BINS5 if int(b[0][:4]) < 2001])]
NOTE5 = (r"Interactions model (D\&K row 4): the author's base model plus union\_x\_outsourced and "
         r"parttime\_x\_outsourced (parttime\_x\_outsourced not shown). A negative Union $\times$ Outsourced "
         r"coefficient means an estimated union wage premium that is smaller for outsourced workers in that bin, not "
         r"a causal claim. Janitors 2021-2025 ($-$0.090, SE 0.030) and guards 1988-1992 ($-$0.102, SE 0.029) are "
         r"significant at $p < .01$; guards 2001-2005 is marginal. Signs vary across bins, with no consistent "
         r"pattern. SEs in parentheses. $^\dagger$1998-2000 and 2016-2019 are shorter bins (3 and 4 years); 2020 is "
         r"excluded from every bin. Data: CPS-ORG via IPUMS.")

def row5(occ, period, short):
    sub = interactions[(interactions["occ"] == occ) & (interactions["period"] == period)].set_index("term")
    cells = [plabel(period, short)]
    for term in ["outsourced", "union", "union_x_outsourced"]:
        r = sub.loc[term]
        cells.append(f"{r['estimate']:.3f} ({r['se']:.3f})")
    cells.append(f"{int(sub.iloc[0]['n_obs']):,}")
    return " & ".join(cells) + r" \\"

header5 = ["Period", "Outsourced", "Union", r"Union $\times$ Outsourced", "N"]
bodyA5 = [r"\toprule", " & ".join(header5) + r" \\", r"\midrule"]
raw5 = []
for panel_label, bins_in_panel in PANELS_A5:
    bodyA5.append(r"\multicolumn{5}{l}{\textbf{" + panel_label + r"}} \\")
    for occ, occ_label in [("janitor", "Janitors"), ("guard", "Guards")]:
        bodyA5.append(r"\multicolumn{5}{l}{\textit{" + occ_label + r"}} \\")
        for label, y0, y1, short in bins_in_panel:
            bodyA5.append(row5(occ, label, short))
    if panel_label == "2001-2025":
        bodyA5.append(r"\addlinespace")
bodyA5.append(r"\bottomrule")

texA5 = "\n".join([
    r"\begin{table}[!t]", r"\centering",
    r"\caption{Table A5: Union Premium by Outsourcing Status, by 5-Year Bin}",
    r"\begin{tabular}{l" + col * 4 + "}",
    *bodyA5,
    r"\end{tabular}",
    r"\begin{minipage}{\linewidth}\footnotesize", NOTE5, r"\end{minipage}",
    r"\end{table}", ""])
with open(f"{OUT}/tables/tableA5_union_premium_interactions.tex", "w") as f:
    f.write(texA5)
interactions.to_csv(f"{OUT}/tables/tableA5_union_premium_interactions.csv", index=False)
print(texA5)